In [1]:
!pip install fastcoref spacy transformers torch networkx tqdm pandas
!python -m spacy download en_core_web_sm
!pip install spacy-entity-linker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 91.8 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
"""
kb_builder_with_coref.py
========================

This script replicates the working kb_builder.ipynb pipeline and adds
cross-passage coreference resolution using fastcoref (FCoref).

Before running:
    pip install fastcoref spacy transformers torch networkx tqdm pandas
    python -m spacy download en_core_web_sm
    pip install spacy-entity-linker

Usage:
    python kb_builder_with_coref.py
"""

import re
import json
import math
import unicodedata
from collections import Counter, defaultdict

import pandas as pd
import networkx as nx
import torch
import spacy
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "Babelscape/rebel-large"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_ARTICLES = 50000      # Set to e.g. 50000 for quick tests
BATCH_SIZE = 32
MIN_REBEL_CONF = 0.35
MIN_GROUND_SCORE = 0.50
EXPORT_GRAPHML = True
EXPORT_JSON = True

# Paths to Wikipedia DPR parquet files (adjust as needed)
PARQUET_PATHS = [
    "hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00000-of-00157.parquet",
    #"hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00001-of-00157.parquet",
    #"hf://datasets/facebook/wiki_dpr/data/psgs_w100/nq/train-00002-of-00157.parquet",
]

# Workflow phase:
#   "coreference"  → Run coreference on ALL articles, save to resolved_articles.json
#   "batch_0"      → Process batch 0 (requires resolved_articles.json)
#   "batch_1"      → Process batch 1 (requires resolved_articles.json)
#   "batch_2"      → Process batch 2 (requires resolved_articles.json)
#   "merge"        → Merge the 3 batch graphs into one final graph
#   "test"         → Quick test on 3 hand-crafted articles
PHASE = "test"
NUM_BATCHES = 3


In [4]:
# =============================================================================
# LOAD MODELS
# =============================================================================

print("Loading fastcoref...")
from fastcoref import FCoref
coref_model = FCoref(device='cpu')
print("fastcoref loaded.")

print("Loading spaCy...")
nlp = spacy.load("en_core_web_sm")
print("Using en_core_web_sm")

print("Adding entity linker...")
try:
    nlp.add_pipe("entityLinker", last=True)
except Exception as e:
    print("Entity linker already added or error:", e)

print("Loading REBEL...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()
print("READY:", DEVICE)


Loading fastcoref...


06/24/2026 22:56:43 - INFO - 	 missing_keys: []
06/24/2026 22:56:43 - INFO - 	 unexpected_keys: []
06/24/2026 22:56:43 - INFO - 	 mismatched_keys: []
06/24/2026 22:56:43 - INFO - 	 error_msgs: []
06/24/2026 22:56:43 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


fastcoref loaded.
Loading spaCy...
Using en_core_web_sm
Adding entity linker...
Loading REBEL...
READY: cuda


In [5]:
# =============================================================================
# GLOBAL STATE
# =============================================================================

LAST_PERSON = None
LAST_ORG = None
LAST_GPE = None

ENTITY_CACHE = {}
PERSON_ALIASES = {}
ORG_ALIASES = {}

G = nx.DiGraph()
REL_COUNTER = Counter()
ENTITY_COUNTER = Counter()
EDGE_STORE = {}


In [6]:
# =============================================================================
# COREFERENCE RESOLUTION (fastcoref)
# =============================================================================

PRONOUNS_SET = {
    "he", "him", "his", "himself",
    "she", "her", "hers", "herself",
    "it", "its", "itself",
    "they", "them", "their", "theirs", "themselves",
    "we", "us", "our", "ours", "ourselves",
    "i", "me", "my", "mine", "myself",
    "you", "your", "yours", "yourself", "yourselves"
}


def resolve_coreference(text: str) -> str:
    """
    Replace pronouns with their representative mentions using fastcoref.
    Handles both (start, end) span tuples and string mention formats.
    """
    if not text or len(text.strip()) < 20:
        return text

    preds = coref_model.predict(texts=[text])
    if not preds or not preds[0].get_clusters():
        return text

    clusters = preds[0].get_clusters()
    if not clusters or not clusters[0]:
        return text

    # Detect cluster format: tuples (spans) vs strings (mentions)
    first_item = clusters[0][0]

    if isinstance(first_item, tuple) and len(first_item) == 2:
        # Format: clusters contain (start, end) index tuples
        replacements = []
        for cluster in clusters:
            if not cluster:
                continue
            rep_start, rep_end = int(cluster[0][0]), int(cluster[0][1])
            representative = text[rep_start:rep_end]
            if len(representative.strip()) < 2:
                continue
            for start, end in cluster:
                start, end = int(start), int(end)
                mention = text[start:end]
                if mention.lower().strip() in PRONOUNS_SET:
                    replacements.append((start, end, representative))

        if not replacements:
            return text

        replacements.sort(key=lambda x: x[0])
        result = []
        last_end = 0
        for start, end, rep in replacements:
            result.append(text[last_end:start])
            result.append(rep)
            last_end = end
        result.append(text[last_end:])
        return "".join(result)

    else:
        # Format: clusters contain mention strings
        for cluster in clusters:
            if not cluster:
                continue
            representative = cluster[0]
            if len(representative.strip()) < 2:
                continue
            for mention in cluster[1:]:
                if mention.lower().strip() in PRONOUNS_SET:
                    text = text.replace(mention, representative, 1)
        return text


In [7]:
# =============================================================================
# DATA LOADING
# =============================================================================

def load_dataset(paths):
    print("\nLOADING DATASET")
    dfs = []
    for p in paths:
        print(f"  Loading {p} ...")
        dfs.append(pd.read_parquet(p))
    df = pd.concat(dfs, ignore_index=True)
    print("Rows:", len(df))

    articles = (
        df.groupby("title")["text"]
        .apply(lambda x: " ".join(x.astype(str)))
        .reset_index()
    )

    if MAX_ARTICLES:
        articles = articles.head(MAX_ARTICLES)

    print("ARTICLES:", len(articles))
    return articles

In [8]:
# =============================================================================
# UTILITIES (unchanged from notebook)
# =============================================================================

def normalize_entity(text):
    if text is None:
        return ""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"<pad>", "", text)
    text = re.sub(r"</?s>", "", text)
    text = re.sub(r"\(.*?\)", "", text)
    text = re.sub(r"\"|\'|`", "", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s\-]", "", text)
    return text.strip()


def update_entity_memory(doc):
    global LAST_PERSON, LAST_ORG, LAST_GPE
    for ent in doc.ents:
        entity = canonicalize_entity(ent.text)
        if ent.label_ == "PERSON":
            LAST_PERSON = entity
            parts = entity.split()
            if len(parts) >= 2:
                PERSON_ALIASES[parts[-1]] = entity
                PERSON_ALIASES[parts[0]] = entity
        elif ent.label_ == "ORG":
            LAST_ORG = entity
            parts = entity.split()
            if len(parts) >= 2:
                ORG_ALIASES[parts[0]] = entity
                ORG_ALIASES[parts[-1]] = entity
        elif ent.label_ in {"GPE", "LOC"}:
            LAST_GPE = entity


def resolve_pronoun(entity):
    entity = normalize_entity(entity)
    if not entity:
        return entity
    if entity in {"he", "him", "his", "she", "her", "hers"}:
        if LAST_PERSON:
            return LAST_PERSON
    if entity in {"it", "its"}:
        if LAST_ORG:
            return LAST_ORG
        if LAST_GPE:
            return LAST_GPE
    if entity in {"they", "them", "their", "theirs"}:
        if LAST_ORG:
            return LAST_ORG
        if LAST_PERSON:
            return LAST_PERSON
    if entity in PERSON_ALIASES:
        return PERSON_ALIASES[entity]
    if entity in ORG_ALIASES:
        return ORG_ALIASES[entity]
    return entity


def canonicalize_entity(entity):
    entity = normalize_entity(entity)
    if not entity:
        return entity
    if entity in ENTITY_CACHE:
        return ENTITY_CACHE[entity]
    try:
        doc = nlp(entity)
        for ent in doc.ents:
            linked = ent._.linkedEntities
            if linked:
                canonical = normalize_entity(linked[0].get_label())
                ENTITY_CACHE[entity] = canonical
                return canonical
    except Exception:
        pass
    return entity


BAD_PRONOUNS = {
    "he", "she", "it", "they", "them", "this", "that",
    "these", "those", "i", "you", "we",
    "who", "which", "where", "when", "what"
}

BLACKLIST_RELATIONS = {
    "be", "have", "do", "become", "take",
    "instance of", "has part", "part of",
    "country", "genre", "occupation", "publication date"
}

BAD_OBJECTS = {
    "yes", "no", "okay", "song", "music", "review", "album",
    "time", "demo", "mixture", "schedule", "show", "tour",
    "event", "episode", "records", "campaign", "talks", "lawyer"
}


def noisy_sentence(sentence):
    s = sentence.lower()
    lyric_patterns = [
        r"\bi love you\b", r"\byou know\b", r"\bme\b",
        r"\bi\b", r"lyrics", r"chorus"
    ]
    if sentence.count('"') >= 4:
        return True
    for p in lyric_patterns:
        if re.search(p, s):
            return True
    return False


VALID_NER = {"PERSON", "ORG", "GPE", "LOC", "WORK_OF_ART", "EVENT", "PRODUCT"}


def parse_sentence(sentence):
    doc = nlp(sentence)
    update_entity_memory(doc)
    ner_entities = set()
    for ent in doc.ents:
        if ent.label_ in VALID_NER:
            ner_entities.add(normalize_entity(ent.text))
    return doc, ner_entities


def subtree_text(token):
    text = " ".join(t.text for t in token.subtree)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_rebel_output(text):
    text = re.sub(r"<pad>", "", text)
    text = text.replace("<s>", "").replace("</s>", "")
    triples = []
    current = {"subject": "", "relation": "", "object": ""}
    state = None
    tokens = text.split()
    for token in tokens:
        if token == "<triplet>":
            if all(current.values()):
                triples.append((
                    current["subject"].strip(),
                    current["relation"].strip(),
                    current["object"].strip()
                ))
            current = {"subject": "", "relation": "", "object": ""}
            state = "subject"
            continue
        elif token == "<subj>":
            state = "object"
            continue
        elif token == "<obj>":
            state = "relation"
            continue
        if state:
            current[state] += token + " "
    if all(current.values()):
        triples.append((
            current["subject"].strip(),
            current["relation"].strip(),
            current["object"].strip()
        ))
    return triples


def grounding_score(entity, sentence, ner_entities):
    e = normalize_entity(entity)
    sent = normalize_entity(sentence)
    score = 0.0
    if e in ner_entities:
        score += 0.65
    if e in sent:
        score += 0.25
    entity_tokens = set(e.split())
    sent_tokens = set(sent.split())
    overlap = len(entity_tokens & sent_tokens)
    if entity_tokens:
        score += (overlap / len(entity_tokens)) * 0.10
    return min(score, 1.0)


def clean_entity(text):
    text = resolve_pronoun(text)
    text = canonicalize_entity(text)
    text = normalize_entity(text)
    if not text:
        return None
    if text in BAD_PRONOUNS:
        return None
    if text in BAD_OBJECTS:
        return None
    if len(text.split()) > 20:
        return None
    return text


def entity_span(token):
    doc = token.doc
    for ent in doc.ents:
        if ent.start <= token.i < ent.end:
            return clean_entity(ent.text)
    subtree_tokens = [t.text for t in token.subtree if t.dep_ != "punct"]
    text = " ".join(subtree_tokens)
    return clean_entity(text)


def dependency_extract(doc, sentence, ner_entities):
    triples = []
    if noisy_sentence(sentence):
        return triples
    for verb in doc:
        if verb.pos_ not in {"VERB", "AUX"}:
            continue
        subj = None
        obj = None
        negated = False
        relation = verb.lemma_
        for child in verb.children:
            if child.dep_ == "neg":
                negated = True
        if negated:
            relation = "NOT_" + relation
        for child in verb.children:
            if child.dep_ in {"nsubj", "nsubjpass"}:
                subj = entity_span(child)
                break
        for child in verb.children:
            if child.dep_ in {"dobj", "attr", "oprd", "obj"}:
                obj = entity_span(child)
                break
        if obj is None:
            for child in verb.children:
                if child.dep_ == "prep":
                    for gc in child.children:
                        if gc.dep_ == "pobj":
                            obj = entity_span(gc)
                            break
                if obj:
                    break
        if not subj or not obj or subj == obj:
            continue
        if relation in BLACKLIST_RELATIONS:
            continue
        subj_score = grounding_score(subj, sentence, ner_entities)
        obj_score = grounding_score(obj, sentence, ner_entities)
        avg_ground = (subj_score + obj_score) / 2
        if avg_ground < MIN_GROUND_SCORE:
            continue
        confidence = (0.60 + avg_ground * 0.35)
        triples.append((subj, relation, obj, round(confidence, 3), "dependency"))
    return triples


def validate_triple(triple):
    s, r, o, conf, src = triple
    s = clean_entity(s)
    r = normalize_entity(r)
    o = clean_entity(o)
    if not s or not o or not r:
        return False
    if s == o:
        return False
    if r in BLACKLIST_RELATIONS:
        return False
    if conf <= 0:
        return False
    return True


def rebel_batch(sentences):
    if not sentences:
        return []
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            return_dict_in_generate=True,
            output_scores=True
        )
    decoded = tokenizer.batch_decode(outputs.sequences, skip_special_tokens=False)
    results = []
    for text, score_tensor in zip(decoded, outputs.sequences_scores):
        beam_score = torch.sigmoid(score_tensor).item()
        results.append((text, beam_score))
    return results


def rebel_confidence(beam_score, subj_ground, obj_ground):
    ground_avg = (subj_ground + obj_ground) / 2
    confidence = (0.55 * beam_score + 0.45 * ground_avg)
    return round(confidence, 3)


BAD_REBEL_RELATIONS = {
    "genre", "instance of", "subclass of",
    "country", "occupation", "publication date"
}


def rebel_hallucination_filter(subj, rel, obj, sentence, ner_entities):
    subj = normalize_entity(subj)
    obj = normalize_entity(obj)
    rel = normalize_entity(rel)
    if rel in BAD_REBEL_RELATIONS:
        return False
    if subj == obj:
        return False
    if subj in BAD_PRONOUNS:
        return False
    if obj in BAD_PRONOUNS:
        return False
    subj_ground = grounding_score(subj, sentence, ner_entities)
    obj_ground = grounding_score(obj, sentence, ner_entities)
    if subj_ground < MIN_GROUND_SCORE:
        return False
    if obj_ground < MIN_GROUND_SCORE:
        return False
    return True


def rebel_extract_batch(sentence_batch, parsed_batch):
    generated = rebel_batch(sentence_batch)
    batch_results = []
    for idx, (sentence, (output, beam_score)) in enumerate(zip(sentence_batch, generated)):
        doc, ner_entities = parsed_batch[idx]
        parsed = parse_rebel_output(output)
        triples = []
        for subj, rel, obj in parsed:
            subj = clean_entity(subj)
            obj = clean_entity(obj)
            rel = normalize_entity(rel)
            if not subj or not obj:
                continue
            if not rebel_hallucination_filter(subj, rel, obj, sentence, ner_entities):
                continue
            subj_ground = grounding_score(subj, sentence, ner_entities)
            obj_ground = grounding_score(obj, sentence, ner_entities)
            conf = rebel_confidence(beam_score, subj_ground, obj_ground)
            if conf < MIN_REBEL_CONF:
                continue
            triples.append((subj, rel, obj, conf, "rebel"))
        batch_results.append(triples)
    return batch_results


def process_sentence_batch(sentence_batch):
    parsed_batch = [parse_sentence(s) for s in sentence_batch]
    rebel_results = rebel_extract_batch(sentence_batch, parsed_batch)
    batch_output = []
    for idx, (sentence, rebel_triples) in enumerate(zip(sentence_batch, rebel_results)):
        doc, ner_entities = parsed_batch[idx]
        dep_triples = dependency_extract(doc, sentence, ner_entities)
        combined = rebel_triples + dep_triples
        cleaned = [triple for triple in combined if validate_triple(triple)]
        batch_output.append(cleaned)
    return batch_output

In [9]:
# =============================================================================
# GRAPH CONSTRUCTION
# =============================================================================

def fuse_confidences(old_conf, new_conf):
    return round(min(1.0, (old_conf * 0.7 + new_conf * 0.3)), 3)


def merge_sources(old_src, new_src):
    sources = set(old_src.split(","))
    sources.add(new_src)
    return ",".join(sorted(sources))


def add_triple_to_graph(graph, triple, article):
    s, r, o, conf, src = triple
    s = resolve_pronoun(s)
    o = resolve_pronoun(o)
    s = canonicalize_entity(s)
    o = canonicalize_entity(o)
    s = normalize_entity(s)
    r = normalize_entity(r)
    o = normalize_entity(o)
    key = (s, r, o)

    if key in EDGE_STORE:
        edge_data = EDGE_STORE[key]
        edge_data["confidence"] = fuse_confidences(edge_data["confidence"], conf)
        edge_data["sources"] = merge_sources(edge_data["sources"], src)
        edge_data["mentions"] += 1
        edge_data["articles"].add(article)
        return False

    EDGE_STORE[key] = {
        "confidence": conf,
        "sources": src,
        "mentions": 1,
        "articles": {article}
    }

    ENTITY_COUNTER[s] += 1
    ENTITY_COUNTER[o] += 1
    REL_COUNTER[r] += 1

    graph.add_node(s, label=s, type="entity")
    graph.add_node(o, label=o, type="entity")
    graph.add_edge(s, o, relation=r, confidence=conf, sources=src, mentions=1, article=article)
    return True


def finalize_graph(graph):
    for (s, r, o), meta in EDGE_STORE.items():
        if not graph.has_edge(s, o):
            graph.add_edge(s, o)
        graph[s][o].update({
            "relation": r,
            "confidence": meta["confidence"],
            "sources": meta["sources"],
            "mentions": meta["mentions"],
            "articles": "; ".join(sorted(meta["articles"]))
        })


In [10]:
# =============================================================================
# EXPORT
# =============================================================================

def export_json(graph, path="semantic_graph_v2.json"):
    data = {"nodes": [], "edges": []}
    for node, attrs in graph.nodes(data=True):
        data["nodes"].append({"id": node, **attrs})
    for u, v, d in graph.edges(data=True):
        data["edges"].append({"source": u, "target": v, **d})
    with open(path, "w", encoding="utf8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print("\nJSON EXPORTED:", path)


def export_graphml(graph, path="semantic_graph_v2.graphml"):
    nx.write_graphml(graph, path)
    print("\nGRAPHML EXPORTED:", path)


In [11]:
# =============================================================================
# BATCHED WORKFLOW HELPERS
# =============================================================================

def save_resolved_articles(articles, path):
    with open(path, "w", encoding="utf8") as f:
        json.dump(articles, f, ensure_ascii=False)
    print(f"  Saved {len(articles)} resolved articles -> {path}")


def load_resolved_articles(path):
    with open(path, "r", encoding="utf8") as f:
        return json.load(f)


def reset_graph_state():
    """Reset all global graph-building state for a fresh batch."""
    global G, EDGE_STORE, REL_COUNTER, ENTITY_COUNTER
    G = nx.DiGraph()
    EDGE_STORE.clear()
    REL_COUNTER.clear()
    ENTITY_COUNTER.clear()


def run_coreference_phase():
    """
    Phase 1: Load dataset, reconstruct articles, run coreference on EVERY article,
    and save resolved articles to disk. Checkpoints every 500 articles.
    """
    articles_df = load_dataset(PARQUET_PATHS)
    resolved_articles = []

    print("\n" + "=" * 80)
    print("PHASE 1: CROSS-PASSAGE COREFERENCE RESOLUTION")
    print("=" * 80)

    for idx, row in tqdm(articles_df.iterrows(), total=len(articles_df)):
        title = row["title"]
        text = row["text"]
        resolved_text = resolve_coreference(text)
        resolved_articles.append({"title": title, "text": resolved_text})

        # Checkpoint every 500 articles
        if idx > 0 and idx % 500 == 0:
            save_resolved_articles(resolved_articles, f"resolved_checkpoint_{idx}.json")

    # Final save
    save_resolved_articles(resolved_articles, "resolved_articles_all.json")
    print(f"\nCoreference phase complete. {len(resolved_articles)} articles saved.")
    print("Next: restart kernel, set PHASE = 'batch_0', and run.")


def process_single_batch(batch_idx):
    """
    Phase 2: Load resolved articles, take one batch, and build a graph.
    Saves the batch graph via pickle (for merging) + JSON/GraphML (for inspection).
    """
    all_articles = load_resolved_articles("resolved_checkpoint_2000.json")

    batch_size = len(all_articles) // NUM_BATCHES
    remainder = len(all_articles) % NUM_BATCHES

    # Distribute remainder across first batches
    start = 0
    for i in range(batch_idx):
        size = batch_size + (1 if i < remainder else 0)
        start += size
    end = start + batch_size + (1 if batch_idx < remainder else 0)
    batch = all_articles[start:end]

    print("\n" + "=" * 80)
    print(f"PHASE 2: PROCESSING BATCH {batch_idx}")
    print(f"  Articles {start} to {end - 1}  ({len(batch)} articles)")
    print("=" * 80)

    reset_graph_state()

    for article in tqdm(batch):
        title = article["title"]
        text = article["text"]

        doc = nlp(text)
        sentences = [
            s.text.strip()
            for s in doc.sents
            if len(s.text.strip()) > 10
        ]

        if not sentences:
            continue

        for i in range(0, len(sentences), BATCH_SIZE):
            sentence_batch = sentences[i:i + BATCH_SIZE]
            results = process_sentence_batch(sentence_batch)
            for sentence, triples in zip(sentence_batch, results):
                for triple in triples:
                    add_triple_to_graph(G, triple, title)

    finalize_graph(G)

    # Save batch outputs
    if EXPORT_JSON:
        export_json(G, f"batch_{batch_idx}_graph.json")
    if EXPORT_GRAPHML:
        export_graphml(G, f"batch_{batch_idx}_graph.graphml")

    # Also pickle the raw graph + edge store for clean merging
    import pickle
    with open(f"batch_{batch_idx}_state.pkl", "wb") as f:
        pickle.dump({"graph": G, "edge_store": dict(EDGE_STORE),
                     "rel_counter": dict(REL_COUNTER),
                     "entity_counter": dict(ENTITY_COUNTER)}, f)
    print(f"  Saved batch state -> batch_{batch_idx}_state.pkl")

    print(f"\nBatch {batch_idx} complete: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print("Restart kernel before processing the next batch.")


def merge_all_batches():
    """
    Phase 3: Load all batch graphs and merge into one final graph.
    """
    import pickle

    print("\n" + "=" * 80)
    print("PHASE 3: MERGING BATCH GRAPHS")
    print("=" * 80)

    reset_graph_state()

    for i in range(NUM_BATCHES):
        pkl_path = f"batch_{i}_state.pkl"
        print(f"\n  Loading {pkl_path} ...")
        with open(pkl_path, "rb") as f:
            state = pickle.load(f)
        batch_G = state["graph"]
        batch_edge_store = state["edge_store"]
        batch_rel = Counter(state["rel_counter"])
        batch_ent = Counter(state["entity_counter"])

        # Merge counters
        REL_COUNTER.update(batch_rel)
        ENTITY_COUNTER.update(batch_ent)

        # Merge nodes
        for node, attrs in batch_G.nodes(data=True):
            if node not in G:
                G.add_node(node, **attrs)

        # Merge edges
        for u, v, attrs in batch_G.edges(data=True):
            if G.has_edge(u, v):
                # Combine metadata
                existing = G[u][v]
                existing["mentions"] = existing.get("mentions", 1) + attrs.get("mentions", 1)
                existing["confidence"] = round(
                    max(existing.get("confidence", 0), attrs.get("confidence", 0)), 3
                )
                # Merge articles string
                existing_arts = set(existing.get("articles", "").split("; "))
                new_arts = set(attrs.get("articles", "").split("; "))
                existing["articles"] = "; ".join(sorted(existing_arts | new_arts))
                # Merge sources
                existing_srcs = set(existing.get("sources", "").split(","))
                new_srcs = set(attrs.get("sources", "").split(","))
                existing["sources"] = ",".join(sorted(existing_srcs | new_srcs))
            else:
                G.add_edge(u, v, **attrs)

        # Merge edge store
        for key, meta in batch_edge_store.items():
            if key in EDGE_STORE:
                EDGE_STORE[key]["mentions"] += meta["mentions"]
                EDGE_STORE[key]["confidence"] = round(
                    max(EDGE_STORE[key]["confidence"], meta["confidence"]), 3
                )
                EDGE_STORE[key]["articles"] = EDGE_STORE[key]["articles"] | meta["articles"]
            else:
                EDGE_STORE[key] = {
                    "confidence": meta["confidence"],
                    "sources": meta["sources"],
                    "mentions": meta["mentions"],
                    "articles": set(meta["articles"]) if isinstance(meta["articles"], list) else meta["articles"]
                }

        print(f"    Batch {i}: merged {batch_G.number_of_nodes()} nodes, {batch_G.number_of_edges()} edges")

    print(f"\nFinal merged graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Export final
    if EXPORT_JSON:
        export_json(G, "semantic_graph_merged.json")
    if EXPORT_GRAPHML:
        export_graphml(G, "semantic_graph_merged.graphml")

    print("\nMERGE COMPLETE.")



In [12]:
def run_test():
    """
    Quick end-to-end test on hand-crafted articles.
    Exercises coreference resolution, REBEL, dependency extraction,
    and graph construction without loading the full dataset.
    """
    print("\n" + "=" * 80)
    print("RUNNING TEST MODE")
    print("=" * 80)

    test_articles = [
        {
            "title": "Albert Einstein",
            "text": (
                "Albert Einstein was a German-born theoretical physicist. "
                "He developed the theory of relativity. Einstein was born in 1879. "
                "He received the Nobel Prize in Physics in 1921. "
                "Einstein was influenced by Isaac Newton and James Clerk Maxwell. "
                "He worked at the Patent Office in Switzerland."
            )
        },
        {
            "title": "Isaac Newton",
            "text": (
                "Isaac Newton was an English mathematician and physicist. "
                "He discovered gravity and wrote Principia Mathematica. "
                "Newton was born in 1643. He influenced Albert Einstein. "
                "He was a key figure in the Scientific Revolution."
            )
        },
        {
            "title": "Marie Curie",
            "text": (
                "Marie Curie was a Polish and naturalized-French physicist and chemist. "
                "She conducted pioneering research on radioactivity. "
                "Curie was the first woman to win a Nobel Prize. "
                "She won the Nobel Prize in Chemistry in 1911. "
                "Her work influenced later research on atomic structure."
            )
        }
    ]

    global G, EDGE_STORE, REL_COUNTER, ENTITY_COUNTER
    G = nx.DiGraph()
    EDGE_STORE.clear()
    REL_COUNTER.clear()
    ENTITY_COUNTER.clear()

    for article in test_articles:
        title = article["title"]
        text = article["text"]

        print(f"\n--- Article: {title} ---")
        print(f"ORIGINAL:\n{text[:200]}...")

        resolved = resolve_coreference(text)
        print(f"\nAFTER COREFERENCE:\n{resolved[:200]}...")

        doc = nlp(resolved)
        sentences = [
            s.text.strip()
            for s in doc.sents
            if len(s.text.strip()) > 10
        ]
        print(f"\nSENTENCES ({len(sentences)}):")
        for s in sentences:
            print(f"  • {s}")

        if sentences:
            results = process_sentence_batch(sentences)
            print(f"\nEXTRACTED TRIPLES:")
            total = 0
            for sentence, triples in zip(sentences, results):
                for triple in triples:
                    subj, rel, obj, conf, src = triple
                    print(f"  [{src}] {subj} --{rel}--> {obj}  (conf={conf})")
                    add_triple_to_graph(G, triple, title)
                    total += 1
            if total == 0:
                print("  (none)")
            else:
                print(f"  Total: {total}")

    finalize_graph(G)

    print("\n" + "=" * 80)
    print("TEST GRAPH SUMMARY")
    print("=" * 80)
    print(f"NODES: {G.number_of_nodes()}")
    print(f"EDGES: {G.number_of_edges()}")

    print("\nTOP RELATIONS")
    for rel, count in REL_COUNTER.most_common(10):
        print(f"  {rel}: {count}")

    print("\nTOP ENTITIES")
    for ent, count in ENTITY_COUNTER.most_common(10):
        print(f"  {ent}: {count}")

    print("\nGRAPH EDGES (sample)")
    sample_count = 0
    for u, v, d in G.edges(data=True):
        print(f"  {u} -- {d['relation']} --> {v}  | conf={round(d['confidence'], 3)} | src={d['sources']} | mentions={d['mentions']}")
        sample_count += 1
        if sample_count >= 20:
            break

    # 5. Export test graph
    if EXPORT_JSON:
        export_json(G, "semantic_graph_test.json")
    if EXPORT_GRAPHML:
        export_graphml(G, "semantic_graph_test.graphml")

    print("\nTEST MODE COMPLETE.")



In [13]:
# =============================================================================
# MAIN PIPELINE
# =============================================================================

def main():
    articles = load_dataset(PARQUET_PATHS)

    print("\nBUILDING GRAPH V2 (with cross-passage coreference)\n")

    for article_idx, row in tqdm(articles.iterrows(), total=len(articles)):
        title = row["title"]
        text = row["text"]

        resolved_text = resolve_coreference(text)

        doc = nlp(resolved_text)
        sentences = [
            s.text.strip()
            for s in doc.sents
            if len(s.text.strip()) > 10
        ]

        if not sentences:
            continue

        for i in range(0, len(sentences), BATCH_SIZE):
            batch = sentences[i:i + BATCH_SIZE]
            results = process_sentence_batch(batch)
            for sentence, triples in zip(batch, results):
                for triple in triples:
                    add_triple_to_graph(G, triple, title)

        if article_idx > 0 and article_idx % 2000 == 0:
            print(f"\nCHECKPOINT {article_idx}")
            export_json(G, f"checkpoint_{article_idx}.json")
            if EXPORT_GRAPHML:
                export_graphml(G, f"checkpoint_{article_idx}.graphml")

    finalize_graph(G)

    print("\n")
    print("=" * 80)
    print("GRAPH SUMMARY V2")
    print("=" * 80)
    print("NODES:", G.number_of_nodes())
    print("EDGES:", G.number_of_edges())

    print("\nTOP RELATIONS")
    for rel, count in REL_COUNTER.most_common(25):
        print(rel, ":", count)

    print("\nTOP ENTITIES")
    for ent, count in ENTITY_COUNTER.most_common(25):
        print(ent, ":", count)

    print("\nGRAPH SAMPLE\n")
    sample_count = 0
    for u, v, d in G.edges(data=True):
        print(u, "--", d["relation"], "-->", v,
              "| conf:", round(d["confidence"], 3),
              "| src:", d["sources"],
              "| mentions:", d["mentions"])
        sample_count += 1
        if sample_count >= 50:
            break

    # Final export
    if EXPORT_GRAPHML:
        export_graphml(G)
    if EXPORT_JSON:
        export_json(G)

    print("\nV2 PIPELINE WITH COREFERENCE COMPLETE.")



In [14]:
if PHASE == "test":
    run_test()
elif PHASE == "coreference":
    run_coreference_phase()
elif PHASE.startswith("batch_"):
    batch_idx = int(PHASE.split("_")[1])
    process_single_batch(batch_idx)
elif PHASE == "merge":
    merge_all_batches()
else:
    print(f"Unknown PHASE: {PHASE}. Use 'coreference', 'batch_0', 'batch_1', 'batch_2', 'merge', or 'test'.")


06/24/2026 22:57:00 - INFO - 	 Tokenize 1 inputs...



RUNNING TEST MODE

--- Article: Albert Einstein ---
ORIGINAL:
Albert Einstein was a German-born theoretical physicist. He developed the theory of relativity. Einstein was born in 1879. He received the Nobel Prize in Physics in 1921. Einstein was influenced by Is...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

06/24/2026 22:57:00 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

...


EXTRACTED TRIPLES:
  [rebel] isaac newton --notable work--> principia mathematica  (conf=0.722)
  [rebel] principia mathematica --author--> isaac newton  (conf=0.722)
  [dependency] isaac newton --discover--> gravity  (conf=0.836)
  [rebel] isaac newton --student--> albert einstein  (conf=0.723)
  [rebel] albert einstein --influenced by--> isaac newton  (conf=0.723)
  [dependency] isaac newton --influence--> albert einstein  (conf=0.95)
  Total: 6

--- Article: Marie Curie ---
ORIGINAL:
Marie Curie was a Polish and naturalized-French physicist and chemist. She conducted pioneering research on radioactivity. Curie was the first woman to win a Nobel Prize. She won the Nobel Prize in Ch...


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

06/24/2026 22:57:10 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]


AFTER COREFERENCE:
Marie Curie was a Polish and naturalized-French physicist and chemist. Marie Curie conducted pioneering research on radioactivity. Curie was the first woman to win a Nobel Prize. Marie Curie won the N...

SENTENCES (5):
  • Marie Curie was a Polish and naturalized-French physicist and chemist.
  • Marie Curie conducted pioneering research on radioactivity.
  • Curie was the first woman to win a Nobel Prize.
  • Marie Curie won the Nobel Prize in Chemistry in 1911.
  • Marie Curie work influenced later research on atomic structure.

EXTRACTED TRIPLES:
  [dependency] marie curie --win--> the nobel prize in chemistry  (conf=0.95)
  Total: 1

TEST GRAPH SUMMARY
NODES: 10
EDGES: 10

TOP RELATIONS
  influenced by: 2
  develop: 1
  receive: 1
  work: 1
  notable work: 1
  author: 1
  discover: 1
  student: 1
  influence: 1
  win: 1

TOP ENTITIES
  isaac newton: 7
  albert einstein: 6
  principia mathematica: 2
  the theory of relativity: 1
  the nobel prize in physics: 1
 